# Week 3 — Synthetic Transaction Data Generator

**目标**：模拟 200 个用户在 30 天内的交易行为，注入 3 类异常模式，保存 parquet 供 Week 4+ 使用。

**产出**：
- `data/synth/transactions.parquet` — 含 `user_id, ts, amount, merchant_cat, lat, lng, device_id, is_anomaly, anomaly_type`
- 可视化：正常 vs 异常在金额/时间/地理维度上的差异

In [ ]:
# ── Bootstrap (Colab + local) ──────────────────────────────────────
# Env detect → Drive mount → Kaggle creds → reproducibility
import os, sys, pathlib, random

IN_COLAB = 'google.colab' in sys.modules
DRIVE_FOLDER = 'LLM/transformer-anomaly'  # edit if your Drive subfolder differs

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    PROJECT_ROOT = pathlib.Path(f'/content/drive/MyDrive/{DRIVE_FOLDER}')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    for sub in ('data', 'models', 'checkpoints'):
        (PROJECT_ROOT / sub).mkdir(exist_ok=True)
    os.chdir(PROJECT_ROOT)

    # Kaggle credentials via Colab Secrets (left sidebar 🔑 icon).
    # Add KAGGLE_USERNAME and KAGGLE_KEY there; never upload kaggle.json.
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    except Exception as e:
        print(f'[bootstrap] Kaggle secrets not configured: {e}')
else:
    PROJECT_ROOT = pathlib.Path.cwd()

# Reproducibility
import numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[bootstrap] env={"colab" if IN_COLAB else "local"}  device={device}')
print(f'[bootstrap] project_root={PROJECT_ROOT}')


## 1. 参数与设计

- **200 用户 × 30 天**：每个用户每天随机 3–8 笔 → 约 30k–48k 行。
- **特征**：
  - `amount` — 每用户 lognormal(μ_u, 1.0)，μ_u 来自用户等级分布（小/中/大消费者）
  - `ts` — 昼夜周期（白天密集、凌晨稀疏），通过 von Mises 混合采样小时
  - `merchant_cat` — 20 类，每用户有 5 类偏好（Dirichlet 分配）
  - `lat/lng` — 围绕用户 home 坐标高斯抖动（σ≈0.05°，~5km）
  - `device_id` — 每用户 1–2 个常用设备

- **3 类异常**（每类 ~1%）：
  1. **金额飙升** — 单笔 > 用户 p99 × 5
  2. **地理跳变** — 两笔间隔 < 1h 但距离 > 1000km
  3. **高频小额突发** — 5 分钟内 >10 笔、每笔 < 用户均值 × 0.1

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
from math import radians, cos, sin, asin, sqrt

N_USERS = 200
N_DAYS = 30
MERCHANT_CATS = [f'cat_{i:02d}' for i in range(20)]
rng = np.random.default_rng(SEED)

# ─── 用户画像 ───
user_tier = rng.choice(['small', 'mid', 'large'], size=N_USERS, p=[0.5, 0.35, 0.15])
mu_map = {'small': 2.5, 'mid': 3.5, 'large': 4.5}   # log amount mean
home_lat = rng.uniform(30, 50, N_USERS)
home_lng = rng.uniform(-120, -70, N_USERS)
# 每用户商户偏好（Dirichlet）
user_cat_pref = rng.dirichlet(alpha=np.ones(20) * 0.3, size=N_USERS)  # 稀疏偏好
user_devices = [[f'dev_{u}_{k}' for k in range(rng.integers(1, 3))] for u in range(N_USERS)]

user_profile = pd.DataFrame({
    'user_id': np.arange(N_USERS),
    'tier': user_tier,
    'mu': [mu_map[t] for t in user_tier],
    'home_lat': home_lat,
    'home_lng': home_lng,
})
print(user_profile.head())
print('tier counts:', user_profile.tier.value_counts().to_dict())

## 2. 正常交易采样

- `n_txn ~ Poisson(5) per user per day`
- `hour ~ 混合 von Mises`（模拟早上/中午/晚上 3 个峰）
- 价格 lognormal，商户按偏好 softmax，位置 home + 高斯抖动。

In [ ]:
def sample_hours(n, rng):
    # 3 peaks at 9, 13, 20 (von Mises-like via normal mix)
    peaks = [9, 13, 20]; stds = [1.5, 1.2, 2.0]; weights = [0.3, 0.3, 0.4]
    choices = rng.choice(3, size=n, p=weights)
    h = np.array([rng.normal(peaks[c], stds[c]) for c in choices])
    return np.mod(h, 24)

rows = []
base_date = pd.Timestamp('2025-01-01')
for u in range(N_USERS):
    mu = user_profile.loc[u, 'mu']
    h_lat, h_lng = user_profile.loc[u, 'home_lat'], user_profile.loc[u, 'home_lng']
    pref = user_cat_pref[u]
    devs = user_devices[u]
    for d in range(N_DAYS):
        n_t = rng.poisson(5)
        if n_t == 0: continue
        hours = sample_hours(n_t, rng)
        for hh in hours:
            minute = rng.integers(0, 60); sec = rng.integers(0, 60)
            ts = base_date + pd.Timedelta(days=int(d), hours=float(hh),
                                          minutes=int(minute), seconds=int(sec))
            amt = float(np.exp(rng.normal(mu, 0.7)))
            cat = MERCHANT_CATS[rng.choice(20, p=pref)]
            lat = float(h_lat + rng.normal(0, 0.05))
            lng = float(h_lng + rng.normal(0, 0.05))
            dev = rng.choice(devs)
            rows.append((u, ts, amt, cat, lat, lng, dev, 0, 'normal'))

df_norm = pd.DataFrame(rows, columns=['user_id','ts','amount','merchant_cat',
                                      'lat','lng','device_id','is_anomaly','anomaly_type'])
print('normal rows:', len(df_norm))
df_norm.head(3)

## 3. 注入异常 A：金额飙升

- 每个用户随机挑 1–3 笔翻成 `user_p99 * uniform(5, 15)`

In [ ]:
p99 = df_norm.groupby('user_id')['amount'].quantile(0.99).to_dict()

mask_A = []
for u in range(N_USERS):
    idx = df_norm.index[df_norm.user_id == u]
    k = rng.integers(1, 4)
    pick = rng.choice(idx, size=min(k, len(idx)), replace=False)
    mask_A.extend(pick.tolist())

df_norm.loc[mask_A, 'amount'] = df_norm.loc[mask_A, 'user_id'].map(p99) * rng.uniform(5, 15, size=len(mask_A))
df_norm.loc[mask_A, 'is_anomaly'] = 1
df_norm.loc[mask_A, 'anomaly_type'] = 'amount_spike'
print('A (amount spike) injected:', len(mask_A))

## 4. 注入异常 B：地理跳变

对每个用户随机选一笔，把它的坐标改成"远离 home 1000km+"的位置，时间戳与前一笔间隔 < 1h。

In [ ]:
def haversine(lat1, lng1, lat2, lng2):
    R = 6371.0
    lat1, lng1, lat2, lng2 = map(radians, [lat1, lng1, lat2, lng2])
    dlat = lat2 - lat1; dlng = lng2 - lng1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlng/2)**2
    return 2 * R * asin(sqrt(a))

df_norm = df_norm.sort_values(['user_id','ts']).reset_index(drop=True)
mask_B = []
for u in range(N_USERS):
    idx = df_norm.index[(df_norm.user_id == u) & (df_norm.is_anomaly == 0)].to_numpy()
    if len(idx) < 2: continue
    pick = int(rng.choice(idx[1:]))  # 不是首笔
    # 远离 home 至少 10 度（≈ 1100km）
    df_norm.at[pick, 'lat'] = df_norm.at[pick, 'lat'] + rng.choice([-1, 1]) * rng.uniform(10, 20)
    df_norm.at[pick, 'lng'] = df_norm.at[pick, 'lng'] + rng.choice([-1, 1]) * rng.uniform(10, 20)
    # 把时间拉到前一笔 + 10~40 分钟
    prev_ts = df_norm.at[pick - 1, 'ts']
    df_norm.at[pick, 'ts'] = prev_ts + pd.Timedelta(minutes=int(rng.integers(10, 40)))
    df_norm.at[pick, 'is_anomaly'] = 1
    df_norm.at[pick, 'anomaly_type'] = 'geo_jump'
    mask_B.append(pick)
print('B (geo jump) injected:', len(mask_B))

## 5. 注入异常 C：高频小额突发

挑 ~2% 的用户，每人一次突发：在某 5 分钟窗口内生成 10–15 笔 < `user_mean * 0.1` 的小额交易。

In [ ]:
burst_users = rng.choice(N_USERS, size=int(N_USERS * 0.2), replace=False)
mean_amt = df_norm[df_norm.is_anomaly == 0].groupby('user_id')['amount'].mean().to_dict()
extra = []
for u in burst_users:
    n = int(rng.integers(10, 16))
    day = int(rng.integers(0, N_DAYS))
    hour = float(rng.uniform(1, 23))
    start_ts = base_date + pd.Timedelta(days=day, hours=hour)
    h_lat, h_lng = user_profile.loc[u, 'home_lat'], user_profile.loc[u, 'home_lng']
    devs = user_devices[u]
    cat = MERCHANT_CATS[rng.choice(20)]
    for i in range(n):
        ts = start_ts + pd.Timedelta(seconds=int(rng.integers(0, 300)))
        amt = float(rng.uniform(0.5, max(1.0, mean_amt[u] * 0.1)))
        extra.append((int(u), ts, amt, cat,
                      float(h_lat + rng.normal(0, 0.02)),
                      float(h_lng + rng.normal(0, 0.02)),
                      str(rng.choice(devs)), 1, 'burst'))

df_burst = pd.DataFrame(extra, columns=df_norm.columns)
df = pd.concat([df_norm, df_burst], ignore_index=True).sort_values(['user_id','ts']).reset_index(drop=True)
print('C (burst) injected:', len(df_burst))
print('total rows:', len(df), '  anomaly rows:', df.is_anomaly.sum(),
      f'({df.is_anomaly.mean()*100:.2f}%)')
df.anomaly_type.value_counts()

## 6. 保存 parquet

In [ ]:
out_dir = PROJECT_ROOT / 'data' / 'synth'
out_dir.mkdir(parents=True, exist_ok=True)
parquet_path = out_dir / 'transactions.parquet'

# dtypes: 把 merchant_cat/device_id 转 category 压缩体积
for c in ['merchant_cat', 'device_id', 'anomaly_type']:
    df[c] = df[c].astype('category')

df.to_parquet(parquet_path, index=False)
user_profile.to_parquet(out_dir / 'users.parquet', index=False)
print('saved:', parquet_path, '  size(MB):', parquet_path.stat().st_size / 1e6)

## 7. 可视化：正常 vs 异常

- 金额分布（log）对比
- 时间（小时）分布对比
- 地理散点（抽样 2k 点）

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# amount log
for typ, color in [('normal', 'C0'), ('amount_spike', 'C3'),
                   ('geo_jump', 'C2'), ('burst', 'C1')]:
    sub = df[df.anomaly_type == typ]['amount']
    if len(sub):
        axes[0].hist(np.log1p(sub), bins=60, alpha=0.5, label=typ, color=color)
axes[0].set_xlabel('log(1+amount)'); axes[0].set_title('Amount distribution by type'); axes[0].legend()

# hour of day
df['hour'] = df.ts.dt.hour
for typ, color in [('normal','C0'),('amount_spike','C3'),('geo_jump','C2'),('burst','C1')]:
    sub = df[df.anomaly_type == typ]
    if len(sub):
        counts = sub.groupby('hour').size()
        counts = counts / max(counts.sum(), 1)
        axes[1].plot(counts.index, counts.values, marker='o', label=typ, color=color)
axes[1].set_xlabel('hour'); axes[1].set_title('Hour-of-day density'); axes[1].legend()

# geo scatter (sampled)
sample = df.sample(n=min(3000, len(df)), random_state=SEED)
for typ, color, m in [('normal','C0','.'),('amount_spike','C3','x'),
                      ('geo_jump','C2','s'),('burst','C1','^')]:
    sub = sample[sample.anomaly_type == typ]
    axes[2].scatter(sub['lng'], sub['lat'], s=8, label=typ, color=color, marker=m, alpha=0.6)
axes[2].set_xlabel('lng'); axes[2].set_ylabel('lat'); axes[2].set_title('Geography (sampled)')
axes[2].legend()
plt.tight_layout()

## 8. 验收

- 总行数 ≈ 30k–50k
- 异常占比 2–5%
- 图中能清晰看到：
  - `amount_spike` 在金额高端独立成峰
  - `geo_jump` 在地理散点远离主簇
  - `burst` 在某些小时密集出现且金额极低

下一步：`03b_sequence_builder.ipynb` 把这份 parquet + Kaggle CSV 都切成 `(N, 32, F)` 的序列张量。